<a href="https://colab.research.google.com/github/melihdural/Gazi_Bilisim_ML/blob/v2.0/Gazi_ML16_ipynb_adl%C4%B1_not_defterinin_kopyas%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datasets

## Model - Dataset Performans Beklentileri

| ID    | Dataset      | Problem Türü / Veri İçeriği | Boyut (Satır x Sütun) | Önerilen Model Tipi | Açıklama |
|------|-------------|-----------------------------|----------------------|---------------------|----------|
| 150  | Covertype   | Coğrafi ve çevresel özelliklerden orman örtüsü tipi sınıflandırma (toprak tipi, yükseklik, eğim, gölge vb.) | ~581K x 54 | XGBoost / LightGBM / CatBoost | Tamamen numeric, yüksek boyutlu ve büyük veri. Sınıflar arası overlap var → kompleks decision boundary gerekir. Boosting modeller üstün. |
| 1590 | Adult       | Bireyin yıllık gelirinin 50K$ üstü/altı tahmini (yaş, eğitim seviyesi, meslek, çalışma saati, medeni durum vb.) | ~48K x 14 | CatBoost | Yoğun kategorik veri (occupation, education, marital-status). Encoding kritik. CatBoost native categorical handling ile avantajlı. |
| 23512| Higgs       | Higgs bozonu tespiti (parçacık çarpışmalarından elde edilen fiziksel ölçümlerle signal/background ayrımı) | ~1M x 28 | XGBoost / LightGBM | Çok büyük ve dense numeric veri. Lineer olmayan kompleks pattern. GPU boosting en iyi performansı verir. |
| 40701| Electricity | Elektrik fiyatının artış/azalış yönü tahmini (zamana bağlı tüketim ve fiyat pattern’leri) | ~45K x 8 | LightGBM / XGBoost | Düşük feature sayısı, zaman bağımlı pattern. Feature interaction önemli. Boosting modeller hızlı ve etkili öğrenir. |
| 40498| Churn       | Müşterinin hizmeti bırakıp bırakmayacağını tahmin (abonelik süresi, kullanım, ödeme davranışı vb.) | ~5K x 20 | Tüm modeller (özellikle boosting) | Küçük veri. Gürültü ve imbalance olabilir. Overfitting riski yüksek → regularization önemli. |
| 40981| Australian  | Kredi başvurusunun onay/red tahmini (finansal durum, kredi geçmişi, demografik özellikler) | 690 x 15 | Tree-based modeller | Küçük ve karışık yapı (numeric + categorical). Veri noisy olabilir. Basit ama stabil modeller avantajlı. |

# Bölüm 1 : Kütüphane Kurulumları

In [ ]:
!pip install catboost lightgbm xgboost pytorch-tabnet flaml scikit-learn pandas numpy matplotlib seaborn shap

# Bölüm 2 : İçe Aktarmalar ve Sabit Tanımlamaları

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import time
import sys
import pickle
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, mean_squared_error, roc_auc_score
)
from sklearn.ensemble import HistGradientBoostingClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

from IPython.display import display


# Seed ve Veri Seti Konfigürasyonları
SEEDS = [42, 123, 2026]
OPENML_DATASET_IDS = [150, 1590, 23512, 40701, 40498, 40981]

# Sonuçları tutacağımız ana liste
results_list = []

# Eğitilen Modeller
trained_models = {}

KeyboardInterrupt: 

# Bölüm 3 : Veri Çekme, Bölme ve EDA Fonksiyonları

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import re

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

def load_and_split_data(dataset_id, seed):
    print(f"--- Veri Seti Yükleniyor: OpenML ID {dataset_id} ---")

    data = fetch_openml(data_id=dataset_id, as_frame=True, parser='auto')
    df = data.frame.copy()

    # --- Target belirleme ---
    target_name = data.target.name if hasattr(data.target, "name") else df.columns[-1]

    X_processed = df.drop(columns=[target_name])
    y_processed = df[target_name]

    # --- Target encode ---
    if y_processed.dtype == 'object' or str(y_processed.dtype).startswith("category"):
        y_processed = LabelEncoder().fit_transform(y_processed)

    # --- Feature temizleme ---
    # sadece numeric al (bilinçli tercih)
    numeric_cols = X_processed.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        raise ValueError(f"Dataset {dataset_id} numeric feature içermiyor.")

    X_processed = X_processed[numeric_cols]

    # --- NaN handling ---
    X_processed = X_processed.fillna(X_processed.median(numeric_only=True))

    # --- Index reset (çok kritik) ---
    X_processed = X_processed.reset_index(drop=True)
    y_processed = pd.Series(y_processed).reset_index(drop=True)

    # --- Split ---
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_processed, y_processed, test_size=0.20, random_state=seed, stratify=y_processed
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.125, random_state=seed, stratify=y_temp
    )

    dataset_name = data.details.get("name", f"DATASET_{dataset_id}").upper()
    dataset_name = f"{dataset_name} | Id: {dataset_id}"

    return X_train, X_val, X_test, y_train, y_val, y_test, X_processed, y_processed, dataset_name

def perform_eda(X, y_target, dataset_id, dataset_name):
    print(f"\n--- OpenML ID {dataset_id} için Veri Analizi (EDA) ---")

    # 🔥 Büyük datasetlerde sample al
    if len(X) > 5000:
        sample_idx = np.random.choice(len(X), 5000, replace=False)
        X_sample = X.iloc[sample_idx]
        y_sample = y_target.iloc[sample_idx]
    else:
        X_sample = X
        y_sample = y_target

    # --- Korelasyon ---
    plt.figure(figsize=(10, 8))
    corr = X_sample.corr()
    sns.heatmap(corr, cmap="coolwarm", annot=False)
    plt.title(f"{dataset_name} - Korelasyon Matrisi")
    plt.tight_layout()
    plt.show()

    # --- Feature dağılım ---
    cols_to_plot = X_sample.columns[:5]

    X_sample[cols_to_plot].hist(
        bins=30,
        figsize=(15, 5),
        layout=(1, len(cols_to_plot))
    )

    plt.suptitle(f"{dataset_name} - Feature Dağılımları (İlk 5)")
    plt.tight_layout()
    plt.show()

    # --- Target dağılım ---
    plt.figure(figsize=(6, 4))
    sns.countplot(x=y_sample)
    plt.title(f"{dataset_name} - Target Distribution")
    plt.tight_layout()
    plt.show()

def perform_model_base_plot(model_name, model, X_train, y_train, X_val, y_val, X_test, y_test, dataset_name):
    y_pred = model.predict(X_test)

    # feature name fallback
    if isinstance(X_train, pd.DataFrame):
        feature_names = X_train.columns
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    # ----------------------------
    # 1) MLP loss curve
    # ----------------------------
    if model_name == "MLP" and hasattr(model, "loss_curve_"):
        plt.figure(figsize=(8, 6))
        plt.plot(model.loss_curve_)
        plt.title(f"{model_name} Loss Curve\n({dataset_name})")
        plt.xlabel("Iteration")
        plt.ylabel("Loss")
        plt.tight_layout()
        plt.show()
        return

    # ----------------------------
    # 2) Feature importance
    # ----------------------------
    if hasattr(model, "feature_importances_"):
        try:
            feat_imp = pd.Series(model.feature_importances_, index=feature_names)
            feat_imp = feat_imp.sort_values(ascending=False).head(10)

            plt.figure(figsize=(8, 6))
            sns.barplot(x=feat_imp.values, y=feat_imp.index)
            plt.title(f"{model_name} Top 10 Feature Importance\n({dataset_name})")
            plt.tight_layout()
            plt.show()
            return
        except Exception:
            pass

    # ----------------------------
    # 3) ROC Curve
    # ----------------------------
    unique_classes = np.unique(y_test)
    is_binary = len(unique_classes) == 2

    if is_binary and hasattr(model, "predict_proba"):
        try:
            y_score = model.predict_proba(X_test)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_score)
            roc_auc = auc(fpr, tpr)

            plt.figure(figsize=(8, 6))
            plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
            plt.plot([0, 1], [0, 1], linestyle="--")
            plt.title(f"{model_name} ROC Curve\n({dataset_name})")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            plt.tight_layout()
            plt.show()
            return
        except Exception:
            pass

    # ----------------------------
    # 4) Confusion Matrix (fallback)
    # ----------------------------
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens")
    plt.title(f"{model_name} Confusion Matrix\n({dataset_name})")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


# Bölüm 4 : Ön İşleme

In [ ]:
def preprocess_with_scaling(X_tr, X_v, X_t, y_tr, y_v, y_t, model_name=None):
    from sklearn.preprocessing import StandardScaler, RobustScaler

    # --- Default: hiçbir şey yapma (tree modeller için) ---
    if model_name is None or model_name in ["XGBoost", "LightGBM", "CatBoost", "GBT_M"]:
        return X_tr, X_v, X_t, y_tr, y_v, y_t

    # --- Neural / TabNet için scaling ---
    if "TabNet" in model_name or model_name == "MLP":
        scaler = RobustScaler()  # 🔥 StandardScaler yerine daha stabil

        X_train = scaler.fit_transform(X_tr)
        X_val   = scaler.transform(X_v)
        X_test  = scaler.transform(X_t)

        return X_train.astype(np.float32), X_val.astype(np.float32), X_test.astype(np.float32), y_tr, y_v, y_t

    # --- fallback ---
    return X_tr, X_v, X_t, y_tr, y_v, y_t

# Bölüm 5 : Model Tanımlamaları

In [ ]:
def get_model_builders(seed, n_classes=2):
    catboost_loss = "Logloss" if n_classes == 2 else "MultiClass"
    xgb_metric = "logloss" if n_classes == 2 else "mlogloss"

    return {
        "XGBoost": lambda: XGBClassifier(
            random_state=seed,
            eval_metric=xgb_metric,
            n_jobs=-1,
            tree_method="hist",
            device="cuda",
            max_depth=6,
            n_estimators=150,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0
        ),

        "LightGBM": lambda: LGBMClassifier(
            random_state=seed,
            device="gpu",
            n_jobs=-1,
            n_estimators=150,
            learning_rate=0.1,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            verbosity=-1,
            min_child_samples=20
        ),

        "CatBoost": lambda: CatBoostClassifier(
            random_state=seed,
            verbose=0,
            task_type="GPU",
            devices="0",
            iterations=150,
            learning_rate=0.1,
            depth=6,
            loss_function=catboost_loss
        ),

        "TabNet": lambda: TabNetClassifier(
            n_d=16,
            n_a=16,
            n_steps=5,
            gamma=1.3,
            seed=seed,
            verbose=0,
            device_name="cuda",
            optimizer_params=dict(lr=2e-3)
        ),

        "GBT_M": lambda: HistGradientBoostingClassifier(
            max_depth=5,
            max_iter=150,
            learning_rate=0.1,
            random_state=seed
        ),
    }

# Bölüm 6 : Model Fit

In [ ]:
def fit_model(
    model_name,
    model,
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    final_fit=False
):

    # -------------------------
    # TABNET (ÖZEL HANDLE)
    # -------------------------
    if "TabNet" in model_name:
        has_val = X_val is not None and y_val is not None

        model.fit(
            X_train=X_train,
            y_train=y_train,
            eval_set=[(X_val, y_val)] if has_val else None,
            eval_metric=["accuracy"] if has_val else None,
            max_epochs=50 if not final_fit else 100,
            patience=10 if has_val else 0,
            batch_size=2048,
            virtual_batch_size=256,
            num_workers=0,
            drop_last=False
      )
        return model

    # -------------------------
    # CATBOOST (GPU + EARLY STOP)
    # -------------------------
    if model_name == "CatBoost":
        if X_val is not None and y_val is not None:
            model.fit(
                X_train,
                y_train,
                eval_set=(X_val, y_val),
                early_stopping_rounds=20,
                verbose=False
            )
        else:
            model.fit(X_train, y_train)

        return model

    # -------------------------
    # LIGHTGBM (GPU + EARLY STOP)
    # -------------------------
    if model_name == "LightGBM":
        if X_val is not None and y_val is not None:
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                eval_metric="logloss",
                callbacks=[]  # 🔥 burada early stopping istersen eklenebilir
            )
        else:
            model.fit(X_train, y_train)

        return model

    # -------------------------
    # XGBOOST (GPU + EARLY STOP)
    # -------------------------
    if model_name == "XGBoost":
        if X_val is not None and y_val is not None:
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        else:
            model.fit(X_train, y_train)

        return model

    # -------------------------
    # DİĞER MODELLER (SKLEARN)
    # -------------------------
    model.fit(X_train, y_train)
    return model

def get_auc_safe(model, X_test, y_test):
    try:
        y_prob = model.predict_proba(X_test)

        # Binary classification
        if y_prob.shape[1] == 2:
            return roc_auc_score(y_test, y_prob[:, 1])

        # Multiclass
        return roc_auc_score(y_test, y_prob, multi_class="ovr")

    except Exception:
        return np.nan

# Bölüm 7: SHAP - XAI

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def perform_academic_shap(model, X_test, feature_names, sample_size=1000, random_state=42):
    print("\n" + "=" * 50)
    print("XAI: SHAP (Shapley Additive exPlanations) Analizi")
    print("=" * 50)

    X_test_np = np.asarray(X_test)

    if len(X_test_np) > sample_size:
        rng = np.random.RandomState(random_state)
        idx = rng.choice(len(X_test_np), sample_size, replace=False)
        X_test_sample = X_test_np[idx]
    else:
        X_test_sample = X_test_np

    if feature_names is None or len(feature_names) != X_test_sample.shape[1]:
        feature_names = [f"f{i}" for i in range(X_test_sample.shape[1])]

    X_test_df = pd.DataFrame(X_test_sample, columns=feature_names)

    try:
        print("SHAP değerleri hesaplanıyor...")
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test_df)

        if isinstance(shap_values, list):
            shap_values_to_plot = shap_values[0]
        elif isinstance(shap_values, np.ndarray):
            shap_values_to_plot = shap_values[:, :, 0] if shap_values.ndim == 3 else shap_values
        else:
            raise ValueError("Beklenmeyen SHAP çıktı formatı.")

        plt.figure(figsize=(10, 6))
        shap.summary_plot(
            shap_values_to_plot,
            X_test_df,
            plot_type="bar",
            show=False
        )
        plt.title("Şekil 1: SHAP Global Özellik Önemi (Bar Plot)", fontsize=14)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 6))
        shap.summary_plot(
            shap_values_to_plot,
            X_test_df,
            show=False
        )
        plt.title("Şekil 2: SHAP Değer Dağılımı ve Etki Yönü (Beeswarm Plot)", fontsize=14)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print("\nHATA: Bu model tipi için TreeExplainer desteklenmiyor veya SHAP hesaplama başarısız oldu.")
        print(f"Hata Detayı: {e}")
        print("İpucu: SHAP analizini tree-based modellerle çalıştır.")

# Bölüm 8 : Ana Eğitim Döngüsü (Training & Cross-Validation)

In [ ]:
for dataset_id in OPENML_DATASET_IDS:

    # --- EDA (1 kez) ---
    _, _, _, _, _, _, X_full_eda, y_full_eda, dataset_name_for_eda = load_and_split_data(dataset_id, SEEDS[0])
    perform_eda(X_full_eda, y_full_eda, dataset_id, dataset_name_for_eda)

    for seed in SEEDS:

        # --- Veri yükleme ---
        X_tr, X_v, X_t, y_tr, y_v, y_t, X_full_loop, y_full_loop, dataset_name_loop = load_and_split_data(dataset_id, seed)

        # --- sınıf sayısı ---
        n_classes = len(np.unique(y_tr))
        model_builders = get_model_builders(seed, n_classes)

        for model_name, model_builder in model_builders.items():

            print(f"Eğitiliyor: {model_name} | Dataset: {dataset_id} | Seed: {seed}")

            # 🔥 MODEL-AWARE PREPROCESS
            X_train, X_val, X_test, y_train, y_val, y_test = preprocess_with_scaling(
                X_tr, X_v, X_t, y_tr, y_v, y_t, model_name=model_name
            )

            # 🔥 numpy dönüşümü burada (model bazlı)
            X_train_np, y_train_np = np.asarray(X_train, dtype=np.float32), np.asarray(y_train)
            X_val_np, y_val_np     = np.asarray(X_val, dtype=np.float32), np.asarray(y_val)
            X_test_np, y_test_np   = np.asarray(X_test, dtype=np.float32), np.asarray(y_test)

            # -------------------------
            # CV
            # -------------------------
            kf = KFold(n_splits=5, shuffle=True, random_state=seed)
            cv_scores = []

            for fold, (train_index, val_index) in enumerate(kf.split(X_train_np), start=1):

                X_kf_train, X_kf_val = X_train_np[train_index], X_train_np[val_index]
                y_kf_train, y_kf_val = y_train_np[train_index], y_train_np[val_index]

                fold_model = model_builder()

                fit_model(
                    model_name=model_name,
                    model=fold_model,
                    X_train=X_kf_train,
                    y_train=y_kf_train,
                    X_val=X_kf_val,
                    y_val=y_kf_val,
                    final_fit=False
                )

                preds_val = fold_model.predict(X_kf_val)
                score = accuracy_score(y_kf_val, preds_val)
                cv_scores.append(score)

            cv_accuracy = float(np.mean(cv_scores))

            # -------------------------
            # FINAL TRAIN
            # -------------------------
            final_model = model_builder()

            fit_model(
                model_name=model_name,
                model=final_model,
                X_train=X_train_np,
                y_train=y_train_np,
                X_val=X_val_np,
                y_val=y_val_np,
                final_fit=True
            )

            # -------------------------
            # TEST
            # -------------------------
            y_pred = final_model.predict(X_test_np)
            auc = get_auc_safe(final_model, X_test_np, y_test_np)

            acc = accuracy_score(y_test_np, y_pred)
            prec = precision_score(y_test_np, y_pred, average="weighted", zero_division=0)
            rec = recall_score(y_test_np, y_pred, average="weighted", zero_division=0)
            f1 = f1_score(y_test_np, y_pred, average="weighted", zero_division=0)
            rmse = np.sqrt(mean_squared_error(y_test_np, y_pred))

            model_size = len(pickle.dumps(final_model))

            # -------------------------
            # PLOT (Sadece 1 seed)
            # -------------------------
            if seed == SEEDS[0]:
                perform_model_base_plot(
                    model_name,
                    final_model,
                    X_train_np,
                    y_train_np,
                    X_val_np,
                    y_val_np,
                    X_test_np,
                    y_test_np,
                    dataset_name_loop
                )

            # -------------------------
            # SONUÇ KAYDI
            # -------------------------
            results_list.append({
                "Dataset": dataset_id,
                "Dataset_Name": dataset_name_loop,
                "Seed": seed,
                "Model": model_name,
                "Model_Size_Bytes": model_size,
                "CV_5Fold_Accuracy": round(cv_accuracy, 4),
                "Test_Accuracy": round(acc, 4),
                "Precision": round(prec, 4),
                "Recall": round(rec, 4),
                "F1_Score": round(f1, 4),
                "AUC": round(auc, 4) if not np.isnan(auc) else "N/A",
                "RMSE": round(rmse, 4)
            })

            # -------------------------
            # SHAP için sakla
            # -------------------------
            trained_models[(dataset_id, seed, model_name)] = {
                "model": final_model,
                "X_test": X_test_np,
                "X_full": X_full_loop,
                "dataset_name": dataset_name_loop
            }

results_df = pd.DataFrame(results_list)

# Bölüm 9 : Sonuç Tablosu ve Modellerin Kıyaslanması

In [ ]:
print("\n" + "="*50)
print("TÜM MODELLERİN METRİK SONUÇLARI")
print("="*50)

pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

# -------------------------
# 1. ORTALAMA + STD
# -------------------------
summary_df = results_df.groupby(['Dataset', 'Model']).agg({
    'Model_Size_Bytes': ['mean'],
    'CV_5Fold_Accuracy': ['mean', 'std'],
    'Test_Accuracy': ['mean', 'std'],
    'Precision': ['mean'],
    'Recall': ['mean'],
    'F1_Score': ['mean'],
    'RMSE': ['mean']
}).reset_index()

# kolon flatten
summary_df.columns = [
    'Dataset', 'Model',
    'Model_Size',
    'CV_Acc_Mean', 'CV_Acc_Std',
    'Test_Acc_Mean', 'Test_Acc_Std',
    'Precision', 'Recall', 'F1_Score', 'RMSE'
]

# -------------------------
# 2. RANKING (DATASET BAZLI)
# -------------------------
summary_df['Rank'] = summary_df.groupby('Dataset')['Test_Acc_Mean']\
                              .rank(ascending=False, method='dense')

# -------------------------
# 3. EN İYİ MODEL SEÇİMİ
# -------------------------
best_models_df = summary_df.loc[
    summary_df.groupby('Dataset')['Test_Acc_Mean'].idxmax()
].reset_index(drop=True)

print("\n--- DATASET BAZLI EN İYİ MODELLER ---")
display(best_models_df)

# -------------------------
# 4. TAM TABLO
# -------------------------
print("\n--- TÜM MODELLER (SIRALI) ---")
display(
    summary_df.sort_values(by=['Dataset', 'Rank'])
)

# -------------------------
# 5. CSV EXPORT
# -------------------------
summary_df.to_csv("Gazi_ML16_Tum_Sonuclar.csv", index=False)
best_models_df.to_csv("Gazi_ML16_En_Iyi_Modeller.csv", index=False)

In [ ]:
from google.colab import sheets
sheet1 = sheets.InteractiveSheet(df=best_models_df)
sheet2 = sheets.InteractiveSheet(df=summary_df)
